# N4 — Validation Notebook
**Competency** C15 · **Artifact** Twin Accuracy Report · **Input** twin log + rig log

Reproduce pos RMSE, settling-time difference, and pressure error (mirrors student/metrics.js twinAccuracy); verify thresholds.


## 1-2. Purpose & inputs
Inputs: `B11-twin-vs-rig.csv` (twin+rig position & pressure). Discrepancy catalogue: `B14-discrepancy-signatures.csv`.


In [1]:
from pathlib import Path
import csv, json, math, urllib.request
# Reference CSVs live in docs/figures. In-repo this resolves directly; on Colab the
# files are pulled from raw GitHub. Students can instead drop their own demo exports in DATA.
REPO_RAW='https://raw.githubusercontent.com/alibulentkoc/parallel-kinematics-hydraulics/main/docs/figures'
DATA = Path('../figures') if Path('../figures').exists() else Path('figures')
DATA.mkdir(exist_ok=True) if DATA==Path('figures') else None
def _ensure(name):
    p=DATA/name
    if not p.exists():
        try: urllib.request.urlretrieve(f'{REPO_RAW}/{name}', p); print('fetched', name)
        except Exception as e: raise FileNotFoundError(f'{name} not found locally and fetch failed: {e}')
    return p
def load_csv(name):
    rows=[]
    with open(_ensure(name)) as f:
        reader=csv.reader(l for l in f if not l.startswith('#'))
        header=next(reader)
        for r in reader:
            if r: rows.append(dict(zip(header,r)))
    return header, rows
def col(rows,k,cast=float):
    return [cast(r[k]) for r in rows if r.get(k) not in (None,'')]


## 3. Reproduce the accuracy metrics


In [2]:
_,b11=load_csv('B11-twin-vs-rig.csv')
tw=col(b11,'twin_x'); rg=col(b11,'rig_x'); twP=col(b11,'twin_P'); rgP=col(b11,'rig_P')
def rmse(a,b): return math.sqrt(sum((p-q)**2 for p,q in zip(a,b))/len(a))
pos_rmse=rmse(tw,rg)*1000
presR=rmse(twP,rgP); mean_rgP=sum(rgP)/len(rgP); pres_err=100*presR/mean_rgP
print(f'pos RMSE {pos_rmse:.2f} mm   pressure error {pres_err:.2f} %')


pos RMSE 0.00 mm   pressure error 0.00 %


## 4. Analyze — pass/fail and (if failing) the discrepancy signature


In [3]:
THR_POS=10.0; THR_PRES=15.0
ok = pos_rmse<=THR_POS and pres_err<=THR_PRES
print('validation:', 'PASS' if ok else 'FAIL')
_,b14=load_csv('B14-discrepancy-signatures.csv')
print('\ndiscrepancy signature catalogue (for diagnosis when a twin fails):')
for r in b14:
    print(f"  {r['fault']:9} RMSE {r['posRMSE_mm']:>5} mm | pErr {r['presErr_pct']:>4} % | ripple {r['holdRipple_mm']:>5} mm | alignHelps {r['alignHelps']}")


validation: PASS

discrepancy signature catalogue (for diagnosis when a twin fails):
  geometry  RMSE   8.0 mm | pErr  0.0 % | ripple   0.1 mm | alignHelps false
  sensor    RMSE   5.4 mm | pErr  0.0 % | ripple   0.1 mm | alignHelps false
  deadband  RMSE   4.2 mm | pErr  0.0 % | ripple  12.1 mm | alignHelps false
  pressure  RMSE   0.0 mm | pErr 18.4 % | ripple   0.1 mm | alignHelps false
  timing    RMSE  13.6 mm | pErr 11.4 % | ripple   0.2 mm | alignHelps true


## 5-6. Generate Twin Accuracy Report + verify


In [4]:
assert ok, 'twin must pass RMSE<=10mm and pressure<=15% for a good twin'
print(f'PASS: RMSE {pos_rmse:.2f} <= {THR_POS} mm and pressure {pres_err:.2f} <= {THR_PRES} %')


PASS: RMSE 0.00 <= 10.0 mm and pressure 0.00 <= 15.0 %


## 7. Export


In [5]:
with open('twin-accuracy.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['metric','value','threshold','pass'])
    w.writerow(['pos_rmse_mm',round(pos_rmse,2),THR_POS,pos_rmse<=THR_POS])
    w.writerow(['pressure_err_pct',round(pres_err,2),THR_PRES,pres_err<=THR_PRES])
print('exported twin-accuracy.csv')


exported twin-accuracy.csv
